In [1]:
['asda', 'sadasda'] + ['adasdasd', 'adas']

['asda', 'sadasda', 'adasdasd', 'adas']

In [ ]:
['dasad']

In [2]:
import socket
import yaml
import pandas as pd
import struct
import time
import os
from tqdm import tqdm
from utils import *

def generate_data(s, config, data_path, Remote_FAULT_LINE, Remote_FAULT_TYPE, Remote_FAULT_DURATION):

    df = pd.DataFrame(columns=config['edge_features']+config['node_features']+config['fault_details'])
    count_iterations = 0
    assert config['fault_trigger_interation'] <= config['sample_size']//2

    while 1:
        trigger_fault = True if count_iterations == config['fault_trigger_interation'] else False

        s.send(struct.pack('>iiiif', config['Remote_Control'],
                            Remote_FAULT_LINE,
                            trigger_fault,
                            config['convert_fault_type(python->rtds)'][Remote_FAULT_TYPE],
                            Remote_FAULT_DURATION))

        recv_data_str = s.recv(config['BUFFER_SIZE'])
        input_format = '>'
        for _ in range(len(config['edge_features'])+len(config['node_features'])):
            input_format += 'f'
        for _ in range(len(config['fault_details'])):
            input_format += 'i'
        recv_data_str_unpacked = list(struct.unpack(input_format, recv_data_str))
        
        # print(recv_data_str_unpacked)
        fault_trip =  recv_data_str_unpacked[-2]
        if not fault_trip: # and not trigger:
            recv_data_str_unpacked[-3] = 8
            recv_data_str_unpacked[-1] = 1
        else:
            recv_data_str_unpacked[-1] = config['convert_fault_type(rtds->python)'][recv_data_str_unpacked[-1]]
            
        df.loc[len(df)] = recv_data_str_unpacked

        count_iterations += 1
        if count_iterations >= config['sample_size']:
            break

    case = get_max_case_number(os.listdir(data_path))+1
    file_name = f"case-{case}-(t{Remote_FAULT_TYPE}_l{Remote_FAULT_LINE}).csv"
    df.to_csv(os.path.join(config['data_path'], config['data_name'], file_name), index=False)
    print(f'Data collected: {file_name}.')

    return

if __name__ == '__main__':
    with open('config.yml', 'r') as c:
        config = yaml.load(c, Loader=yaml.FullLoader)

    data_path = os.path.join(config['data_path'], config['data_name'])
    os.makedirs(data_path, exist_ok=True)

    # s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    # s.connect((config['TCP_IP'], config['TCP_PORT']))
    print('RTDS connected!')

    # for fl in tqdm(config['Remote_FAULT_LINE'], desc='Simulation Process'):
    #     for ft in config['Remote_FAULT_TYPE']:
    #         for fd in config['Remote_FAULT_DURATION']:
    #             generate_data(s, config, data_path, fl, ft, fd)

    variables = {k:None for k in config['variables'].keys()}
    variable_index = {k:0 for k in config['variables'].keys()}

    case_count = 1
    for k,v in config['variables']:
        case_count *= len(v)

    i = 0
    while i <= case_count:
        for k,v in config['variables']:
            if variable_index[k] >= len(v):
                continue
            variables[k] = v[variable_index[k]]
            variable_index[k] += 1
        print(i+1, variables)
        i += 1

        if i >= case_count:
            break

    time.sleep(1)   #This sleep is needed for the ClosePort() below
    # s.close()
    print (f'Data Recieved!!')


RTDS connected!


ValueError: too many values to unpack (expected 2)